# Mini-TP 3 — Sirve tu modelo por gRPC (Sesión 3)

Expone **mi modelo** (`iris-rf`, el mismo servido por REST en el Mini-TP 1 y
descripto por GraphQL en el Mini-TP 2) como un servicio **gRPC**, con un
contrato tipado (`proto/scoring.proto`), un método **unary** y uno de
**server-streaming**, y compara la latencia contra el mismo endpoint REST
(`POST /v1/predict`).

Servidor y stubs viven en `src/model_service/` (`grpc_server.py`,
`scoring_pb2*.py`), compartiendo la carga del modelo con REST y GraphQL vía
`model.py`.

## 1. El contrato `.proto`

Definido en `src/model_service/proto/scoring.proto` — mismos 4 features que `PredictRequest` de REST:

In [1]:
print(open("../src/model_service/proto/scoring.proto").read())

syntax = "proto3";

package scoring;

// Mismos 4 features reales del modelo iris-rf (ver PredictRequest en rest.py).
message Features {
  double sepal_length_cm = 1;
  double sepal_width_cm = 2;
  double petal_length_cm = 3;
  double petal_width_cm = 4;
}

message FeatureBatch {
  repeated Features items = 1;
}

message Prediction {
  int32 prediction = 1;
  string prediction_label = 2;
  string model_version = 3;
}

service Scoring {
  // Unary: una predicción por llamada (equivalente a POST /v1/predict).
  rpc Predict (Features) returns (Prediction);
  // Server-streaming: puntúa un lote y devuelve las predicciones como stream.
  rpc PredictBatch (FeatureBatch) returns (stream Prediction);
}



## 2. Stubs generados

```bash
uv run python -m grpc_tools.protoc -I./src/model_service/proto \
    --python_out=./src/model_service --grpc_python_out=./src/model_service \
    ./src/model_service/proto/scoring.proto
```

Genera `scoring_pb2.py` (mensajes) y `scoring_pb2_grpc.py` (stub cliente +
clase base del servidor) dentro del paquete `model_service`.

In [2]:
import os
print(sorted(f for f in os.listdir("../src/model_service") if f.startswith("scoring_pb2")))

['scoring_pb2.py', 'scoring_pb2_grpc.py']


## 3. El servidor (`grpc_server.py`)

Carga el modelo **una sola vez** (reutiliza `model.py`, igual que REST y
GraphQL) y expone:
- `Predict` (unary) — una predicción, equivalente a `POST /v1/predict`.
- `PredictBatch` (server-streaming) — puntúa un lote y devuelve las
  predicciones como stream.

Lo levanto en un hilo, junto con el REST del Mini-TP 1, para comparar ambos.

In [3]:
import threading, time, socket
import grpc, uvicorn

from model_service import grpc_server as gs
from model_service import scoring_pb2, scoring_pb2_grpc
from model_service.rest import app as rest_app

def serve_rest(app, port):
    def _run():
        uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port,
                                      log_level="warning")).run()
    threading.Thread(target=_run, daemon=True).start()
    for _ in range(50):
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=0.2):
                return
        except OSError:
            time.sleep(0.2)
    raise RuntimeError(f"el servidor en :{port} no respondió")

serve_rest(rest_app, 8100)
REST = "http://127.0.0.1:8100"

servidor_grpc = gs.serve(port=50051)
time.sleep(0.3)
canal = grpc.insecure_channel("localhost:50051")
stub = scoring_pb2_grpc.ScoringStub(canal)

print("REST ->", REST)
print("gRPC ->", "localhost:50051")

REST -> http://127.0.0.1:8100
gRPC -> localhost:50051


## 4. Cliente unary — una predicción

In [4]:
entrada = scoring_pb2.Features(
    sepal_length_cm=5.1, sepal_width_cm=3.5, petal_length_cm=1.4, petal_width_cm=0.2,
)
r = stub.Predict(entrada)
print(f"prediction={r.prediction} label={r.prediction_label!r} version={r.model_version}")

prediction=0 label='setosa' version=1.0.0


## 5. Cliente streaming — puntuar un lote (`PredictBatch`)

In [5]:
lote = scoring_pb2.FeatureBatch(items=[
    scoring_pb2.Features(sepal_length_cm=5.1, sepal_width_cm=3.5, petal_length_cm=1.4, petal_width_cm=0.2),  # setosa
    scoring_pb2.Features(sepal_length_cm=6.0, sepal_width_cm=2.9, petal_length_cm=4.5, petal_width_cm=1.5),  # versicolor
    scoring_pb2.Features(sepal_length_cm=6.7, sepal_width_cm=3.1, petal_length_cm=5.6, petal_width_cm=2.4),  # virginica
])
for p in stub.PredictBatch(lote):
    print(f"{p.prediction:>2}  {p.prediction_label:<12} v{p.model_version}")

 0  setosa       v1.0.0
 1  versicolor   v1.0.0
 2  virginica    v1.0.0


## 6. Comparación de latencia: gRPC vs REST

In [6]:
import requests

N = 200
feat_grpc = scoring_pb2.Features(sepal_length_cm=5.1, sepal_width_cm=3.5, petal_length_cm=1.4, petal_width_cm=0.2)
feat_rest = {"sepal_length_cm": 5.1, "sepal_width_cm": 3.5, "petal_length_cm": 1.4, "petal_width_cm": 0.2}

t = time.perf_counter()
for _ in range(N):
    stub.Predict(feat_grpc)
grpc_ms = (time.perf_counter() - t) / N * 1000

session = requests.Session()
t = time.perf_counter()
for _ in range(N):
    session.post(f"{REST}/v1/predict", json=feat_rest)
rest_ms = (time.perf_counter() - t) / N * 1000

print(f"gRPC: {grpc_ms:.3f} ms/llamada")
print(f"REST: {rest_ms:.3f} ms/llamada")
print(f"gRPC ~{rest_ms/grpc_ms:.1f}x más rápido que REST")

gRPC: 16.535 ms/llamada
REST: 16.800 ms/llamada
gRPC ~1.0x más rápido que REST


### Reflexión — gRPC vs REST

- **Latencia medida:** gRPC y REST quedaron prácticamente empatados (16.54 ms
  vs 16.80 ms por llamada, ~1.0x), muy lejos de la ventaja que muestra el lab
  teórico con modelos de juguete. La razón: acá el modelo es un
  `RandomForestClassifier` de 200 árboles (no una regresión logística), así
  que la **inferencia domina el tiempo total** y el ahorro de transporte de
  gRPC (HTTP/2 multiplexado + Protobuf binario vs JSON/HTTP-1.1) queda diluido
  frente al costo del modelo. En localhost, con ambos servidores compitiendo
  por el mismo CPU, esa diferencia de transporte deja de notarse; bajo mayor
  concurrencia o red real la brecha relativa a favor de gRPC crecería.
- **Cuándo gRPC:** comunicación *interna* entre servicios de la plataforma
  (por ejemplo un gateway llamando al servicio de inferencia) donde el volumen
  de requests es alto y cada milisegundo de *transporte* cuenta, o cuando
  necesito streaming real (`PredictBatch` puntuando un lote sin esperar a que
  termine todo antes de recibir el primer resultado).
- **Cuándo REST:** la API pública de la plataforma. Cualquier cliente HTTP
  (navegador, `curl`, Postman) la habla sin generar nada; no hace falta
  distribuir `.proto`/stubs.
- **Costo de gRPC:** contrato estricto (`.proto`) que hay que versionar y
  regenerar (`scoring_pb2*.py`) cada vez que cambia; no lo habla un navegador
  directamente (necesita grpc-web o un gateway); curva de aprendizaje más alta
  que REST/GraphQL para quien recién arranca. Acá el costo del contrato no se
  justificó en latencia pura, pero sí en la interfaz **estrictamente tipada**
  y en tener listo el método de streaming para lotes.

In [7]:
# limpieza
canal.close()
servidor_grpc.stop(0)
print("Servidor gRPC detenido.")

Servidor gRPC detenido.
